## Setup — Delta tables under a real Unity Catalog schema
Moving from Parquet-in-a-Volume to managed Delta tables under workspace.bronze. This unlocks three things at once: time travel (DESCRIBE HISTORY, demonstrated at the end), automatic Unity Catalog lineage between tables, and COMMENT metadata visible in Catalog Explorer. Raw CSVs stay in the Volume unchanged — that's still the correct use of a Volume, it's only the output that changes.

In [0]:
from pyspark.sql.types import StructField, StringType, StructType
from pyspark.sql import functions as F

CATALOG = "workspace"
RAW_SCHEMA = "transfermarkt"   # the Volume with raw CSVs — unchanged
BRONZE_SCHEMA = "bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

## The 11 tables that ingest cleanly
Each now saves as a managed Delta table with a table-level comment, instead of a Parquet write to a Volume path.

In [0]:
tables = [
    "competitions", "clubs", "players", "appearances",
    "player_valuations", "club_games", "game_events",
    "game_lineups", "transfers", "countries", "national_teams",
]

for t in tables:
    raw_path = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/raw/{t}.csv"
    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{t}")
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.{t} IS "
              f"'Raw {t} data, typed but unmodified from source CSV'")
    print(f"{t}: {df.count()} rows")

## games, handled separately because of aggregate
Same fix as always — aggregate kept as text so it isn't misread as a time value — now saved as Delta.

In [0]:
raw_path = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/raw/games.csv"

inferred = spark.read.csv(raw_path, header=True, inferSchema=True).schema
fixed_schema = StructType([
    StructField(f.name, StringType(), f.nullable) if f.name == "aggregate" else f
    for f in inferred.fields
])

games = spark.read.csv(raw_path, header=True, schema=fixed_schema)
games.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.games")
spark.sql(f"COMMENT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.games IS "
          f"'Raw games data; aggregate kept as text, not inferred as a time value'")
print(f"games: {games.count()} rows")

## Verify the fix, reading as a table now instead of a Parquet path

In [0]:
def bronze(t):
    return spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{t}")

bronze("games").select("aggregate").filter(F.col("aggregate").isNotNull()).show(5)

## Time travel, demonstrated
DESCRIBE HISTORY shows every version of the table — every overwrite from here forward is now a recoverable point in time, not just the latest state. If this project ever needs another fix like the aggregate bug, the pre-fix version would now actually be queryable.

In [0]:
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.games").show(truncate=False)